# Exercise: Building a Mini-GPT

Welcome! In this exercise, you will implement the core components of a GPT-style Transformer model. You'll build the Causal Self-Attention mechanism and then use it to construct a complete Transformer block. This will give you a hands-on understanding of how the architecture works from the inside out.

**In this exercise you will:**
- Implement a Causal Self-Attention layer (`MiniAttention`) from scratch.
- Construct a Transformer `EncoderBlock` by combining your attention layer with a feed-forward network and residual connections.
- Assemble these components into a basic `MiniGPT` model and verify its architecture.

Let's get started!

In [ ]:
import math
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass

#
# --- Pre-written Code: Students should NOT modify this cell ---
#

@dataclass
class ModelConfig:
    """Configuration class for MiniGPT, similar to what you saw in the lecture."""
    context_size: int = 64
    vocab_size: int = 512
    embed_dim: int = 128
    num_blocks: int = 4
    num_heads: int = 4

# A simple helper to make sure our tests are deterministic
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

print("Setup complete. Let's build a Transformer!")

### Exercise 1: Causal Self-Attention

Your first task is to implement the `forward` pass for the `MiniAttention` module. This is the heart of the Transformer!

You will implement scaled dot-product attention with a causal mask to prevent the model from "looking into the future".

**Your implementation should follow these steps:**
1.  **Project Inputs**: Use the pre-defined `self.c_attn` layer to project the input `x` into a combined Query, Key, and Value tensor.
2.  **Split Q, K, V**: Split the combined tensor from the previous step into separate Q, K, and V tensors. Remember that `c_attn` produces them concatenated along the last dimension.
3.  **Reshape for Multi-Head**: Reshape Q, K, and V so that the embedding dimension is split across the attention heads. The final shape for each should be `(B, num_heads, T, head_size)`, where `B` is batch size, `T` is sequence length, and `head_size = embed_dim // num_heads`.
4.  **Compute Attention Scores**: Calculate the raw attention scores by performing a scaled matrix multiplication of Q and K (`Q @ K.transpose(-2, -1)`). Don't forget to scale by `1 / sqrt(head_size)`.
5.  **Apply Causal Mask**: Apply the causal mask (`self.bias`) to the attention scores. The mask is pre-computed for you in the `__init__` method. It ensures that a position can only attend to itself and previous positions.
6.  **Softmax**: Apply a softmax function to the masked scores to get the final attention weights.
7.  **Apply to V**: Multiply the attention weights by the Value tensor (V).
8.  **Combine Heads & Project**: Reshape the output back to `(B, T, C)` and pass it through the final projection layer, `self.c_proj`.

**Hint**: The `torch.Tensor.split()` and `torch.Tensor.view()` or `torch.Tensor.reshape()` methods will be very useful here. Pay close attention to the tensor shapes at each step!

In [ ]:
class MiniAttention(nn.Module):
    """A single head of self-attention."""

    def __init__(self, config: ModelConfig):
        super().__init__()
        assert config.embed_dim % config.num_heads == 0
        # Key, Query, Value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.embed_dim, 3 * config.embed_dim, bias=False)
        # Output projection
        self.c_proj = nn.Linear(config.embed_dim, config.embed_dim, bias=False)
        # Store config values
        self.num_heads = config.num_heads
        self.embed_dim = config.embed_dim
        self.head_size = self.embed_dim // self.num_heads
        
        # A causal mask to ensure attention is only applied to the left in the input sequence
        self.register_buffer("bias", torch.tril(torch.ones(config.context_size, config.context_size))
                                     .view(1, 1, config.context_size, config.context_size))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for Causal Self-Attention.
        
        Args:
            x (torch.Tensor): Input tensor of shape (B, T, C)
                              B = batch size, T = sequence length, C = embedding dimension.
        
        Returns:
            torch.Tensor: Output tensor of shape (B, T, C)
        """
        B, T, C = x.size() # Batch size, sequence length, embedding dimensionality

        ### START CODE HERE ### (≈ 10-12 lines)
        
        # 1. & 2. Calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k, v  = self.c_attn(x).split(self.embed_dim, dim=2)
        
        # 3. Reshape Q, K, V for multi-head attention
        # (B, T, C) -> (B, T, num_heads, head_size) -> (B, num_heads, T, head_size)
        k = k.view(B, T, self.num_heads, self.head_size).transpose(1, 2) 
        q = q.view(B, T, self.num_heads, self.head_size).transpose(1, 2) 
        v = v.view(B, T, self.num_heads, self.head_size).transpose(1, 2) 

        # 4. & 5. Causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        
        # 6. Apply softmax
        att = F.softmax(att, dim=-1)
        
        # 7. Apply to V
        y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        
        # 8. Re-assemble all head outputs side by side
        y = y.transpose(1, 2).contiguous().view(B, T, C) 
        
        # Output projection
        y = self.c_proj(y)
        
        ### END CODE HERE ###
        
        return y

In [ ]:
# --- Test Cell for Exercise 1 ---
set_seed()
config = ModelConfig(context_size=8, embed_dim=16, num_heads=4)
attention_layer = MiniAttention(config)

# Create a dummy input tensor
B, T, C = 2, 5, 16 # Batch, Time, Channels(embed_dim)
dummy_x = torch.randn(B, T, C)

# Get the output
output = attention_layer(dummy_x)

# --- Assertions ---
# 1. Test output shape
expected_shape = (B, T, C)
assert output.shape == expected_shape, f"Wrong output shape! Got {output.shape}, expected {expected_shape}"

# 2. Test output values for a known input
# This ensures the masking, softmax, and projections are likely correct.
expected_first_val = 0.0459
actual_first_val = output[0, 0, 0].item()
assert abs(actual_first_val - expected_first_val) < 1e-4, f"Value mismatch. Expected first element to be ~{expected_first_val}, but got {actual_first_val}."

expected_last_val = -0.0153
actual_last_val = output[-1, -1, -1].item()
assert abs(actual_last_val - expected_last_val) < 1e-4, f"Value mismatch. Expected last element to be ~{expected_last_val}, but got {actual_last_val}."

# 3. Test the causal mask logic
# The attention weights for a given token should be zero for all subsequent tokens.
# We can check this by accessing the attention weights (by slightly modifying the class or just re-calculating here)
q, k, v = attention_layer.c_attn(dummy_x).split(attention_layer.embed_dim, dim=2)
B, T, C = dummy_x.size()
k = k.view(B, T, attention_layer.num_heads, attention_layer.head_size).transpose(1, 2)
q = q.view(B, T, attention_layer.num_heads, attention_layer.head_size).transpose(1, 2)
att_scores = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
att_scores = att_scores.masked_fill(attention_layer.bias[:,:,:T,:T] == 0, float('-inf'))
att_weights = F.softmax(att_scores, dim=-1)

# For any token `i`, the attention weight for token `j` where `j > i` should be 0.
assert torch.allclose(att_weights[0, 0, 2, 3], torch.tensor(0.0)), "Causal mask failed! Token 2 should not attend to token 3."
assert att_weights[0, 0, 2, 0] > 0, "Causal mask failed! Token 2 should attend to token 0."

print("✅ All tests for Exercise 1 passed! Great job implementing the attention mechanism.")

### Exercise 2: The Transformer Encoder Block

Excellent! Now that you have a working attention layer, you'll combine it with the other essential components to build a full Transformer `EncoderBlock`.

A standard Transformer block (using the "pre-norm" formulation popular in GPT models) consists of:
1.  **Layer Normalization**: Applied to the input before the attention layer.
2.  **Causal Self-Attention**: The `MiniAttention` module you just built.
3.  **Residual Connection**: The output of the attention layer is added back to the original input (`x`).
4.  **Another Layer Normalization**: Applied to the result of the first residual connection.
5.  **Feed-Forward Network (MLP)**: A simple multi-layer perceptron.
6.  **Another Residual Connection**: The output of the MLP is added to its input.

Your task is to implement the `forward` pass for the `EncoderBlock` class below, connecting these components in the correct order.

In [ ]:
class EncoderBlock(nn.Module):
    """A single Transformer block."""

    def __init__(self, config: ModelConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.embed_dim)
        self.attn = MiniAttention(config)
        self.ln_2 = nn.LayerNorm(config.embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(config.embed_dim, 4 * config.embed_dim),
            nn.GELU(),
            nn.Linear(4 * config.embed_dim, config.embed_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for the EncoderBlock.
        
        Args:
            x (torch.Tensor): Input tensor of shape (B, T, C)
        
        Returns:
            torch.Tensor: Output tensor of shape (B, T, C)
        """
        ### START CODE HERE ### (≈ 2 lines)
        
        # 1. Layer Norm, Attention, and first Residual Connection
        x = x + self.attn(self.ln_1(x))
        
        # 2. Layer Norm, MLP, and second Residual Connection
        x = x + self.mlp(self.ln_2(x))
        
        ### END CODE HERE ###
        
        return x

In [ ]:
# --- Test Cell for Exercise 2 ---
set_seed()
config = ModelConfig(context_size=8, embed_dim=16, num_heads=4)
encoder_block = EncoderBlock(config)

# Create a dummy input tensor
B, T, C = 2, 5, 16 # Batch, Time, Channels
dummy_x = torch.randn(B, T, C)

# Get the output
output = encoder_block(dummy_x)

# --- Assertions ---
# 1. Test output shape
expected_shape = (B, T, C)
assert output.shape == expected_shape, f"Wrong output shape! Got {output.shape}, expected {expected_shape}"

# 2. Test output values for a known input
expected_first_val = 1.9427
actual_first_val = output[0, 0, 0].item()
assert abs(actual_first_val - expected_first_val) < 1e-4, f"Value mismatch. Expected first element to be ~{expected_first_val}, but got {actual_first_val}."

expected_last_val = -0.5891
actual_last_val = output[-1, -1, -1].item()
assert abs(actual_last_val - expected_last_val) < 1e-4, f"Value mismatch. Expected last element to be ~{expected_last_val}, but got {actual_last_val}."

print("✅ All tests for Exercise 2 passed! You have successfully built a Transformer block.")

### Challenge (Optional): Add Dropout

For an extra challenge, you can add dropout for regularization, which is crucial for training larger models.

Modify your `MiniAttention` and `EncoderBlock` implementations to include dropout layers.
- In `MiniAttention`, add dropout after the softmax and after the final projection (`c_proj`).
- In `EncoderBlock`, add dropout at the end of the MLP.
- Add a `dropout: float = 0.1` parameter to your `ModelConfig` and use it to initialize `nn.Dropout(config.dropout)`.

This is an optional exercise; you don't need to complete it to continue. The integration test below will work with your non-dropout version.

In [ ]:
# --- Integration Test: Assembling the MiniGPT Model ---

# This final cell brings everything together.
# We provide the full MiniGPT model structure, which uses your EncoderBlock.
# Your goal is just to run this cell and see that your components integrate correctly.
# No code needs to be written here.

class MiniGPT(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.embed_dim),
            wpe = nn.Embedding(config.context_size, config.embed_dim),
            h = nn.ModuleList([EncoderBlock(config) for _ in range(config.num_blocks)]),
            ln_f = nn.LayerNorm(config.embed_dim),
        ))
        self.lm_head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.size()
        assert T <= self.config.context_size, f"Cannot forward sequence of length {T}, block size is only {self.config.context_size}"
        
        # Forward the GPT model itself
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device) # shape (T)
        
        # Get token and position embeddings
        tok_emb = self.transformer.wte(idx) # (B, T, C)
        pos_emb = self.transformer.wpe(pos) # (T, C)
        x = tok_emb + pos_emb
        
        # Pass through the transformer blocks
        for block in self.transformer.h:
            x = block(x)
            
        # Final layer norm and language model head
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x) # (B, T, vocab_size)
        
        return logits

# --- Run the integration test ---
set_seed()
config = ModelConfig()
model = MiniGPT(config)

# Create a dummy input (batch of token indices)
dummy_indices = torch.randint(0, config.vocab_size, (2, 10)) # (B, T)

# Get the model output (logits)
logits = model(dummy_indices)

# --- Assertion ---
expected_shape = (dummy_indices.shape[0], dummy_indices.shape[1], config.vocab_size)
assert logits.shape == expected_shape, f"Final model output has wrong shape! Got {logits.shape}, expected {expected_shape}"

print("🎉 Congratulations! Your components have been successfully integrated into a MiniGPT model.")
print("You've built the core architecture of a GPT-style Transformer from scratch.")